# 🚀 LegendStack Tutorial

Welcome to the **LegendStack Agentic AI Framework** interactive tutorial!

This notebook will guide you through building production-ready AI agents, step by step.

## What You'll Learn

| Chapter | Topic | Duration |
|---------|-------|---------|
| 1 | Hello LegendStack | 5 min |
| 2 | Understanding the Graph | 10 min |
| 3 | Safety Guardrails | 10 min |
| 4 | RAG Deep Dive | 15 min |
| 5 | Graph-RAG | 15 min |
| 6 | Memory & Entities | 10 min |
| 7 | Self-Correction | 10 min |
| 8 | Build Your Agent | 20 min |

**Total estimated time: ~90 minutes**

---

## Prerequisites

Before starting, make sure you have:

1. Python 3.11+
2. The LegendStack repository cloned
3. Dependencies installed (`uv sync`)

```bash
# Quick setup
git clone https://github.com/LegendStack/agentic-fastapi-template
cd agentic-fastapi-template
uv sync
```

---

# Chapter 1: Hello LegendStack 👋

**Learning Objectives:**
- Import and instantiate the Demo Agent
- Send your first message
- Understand the response structure

The Demo Agent is a fully-featured agent that runs with **zero external dependencies** using mock services.

In [ ]:
# First, let's set up our path (run this first!)
import sys

sys.path.insert(0, "../src")

# Import the Demo Agent
from app.agents.demo import DemoAgentConfig, LegendDemoAgent

print("✅ LegendStack imported successfully!")

In [ ]:
# Create a Demo Agent with all features enabled
agent = LegendDemoAgent()

# See what features are enabled
print("📋 Feature Configuration:")
for feature, enabled in agent.get_config_summary().items():
    status = "✅" if enabled else "❌"
    print(f"   {status} {feature}")

In [ ]:
# Send your first message!
result = await agent.chat("What is LegendStack?", thread_id="tutorial-1")

print("🤖 Response:")
print(result["response"])
print()
print("📊 Features Used:", result["features_used"])

### Understanding the Response

The agent returns a dictionary with:

| Key | Description |
|-----|-------------|
| `response` | The AI-generated answer |
| `features_used` | Which framework features were invoked |
| `cache_hit` | Whether the response came from cache |
| `cost_info` | Token usage and cost estimates |
| `entities_extracted` | Any entities discovered in the input |

### 💡 Try It Yourself

Modify the query below and observe how the features used change:

In [ ]:
# TRY: Change this query!
your_query = "How does RAG work?"

result = await agent.chat(your_query, thread_id="tutorial-1")
print(f"Query: {your_query}")
print(f"Features: {result['features_used']}")
print(f"Cache Hit: {result['cache_hit']}")

---

# Chapter 2: Understanding the Graph 🔀

**Learning Objectives:**
- Understand LangGraph workflow structure
- Visualize the agent's decision flow
- Learn about conditional routing

LegendStack uses **LangGraph** to orchestrate complex agent workflows.

### The Demo Agent Flow

```mermaid
graph TD
    START([Start]) --> INPUT[Input Node<br/>PII & Moderation]
    INPUT --> CACHE{Cache Check}
    
    CACHE -->|Hit| OUTPUT[Output Node]
    CACHE -->|Miss| RAG[RAG Node<br/>Vector Search]
    
    RAG --> GRAPH_RAG[Graph-RAG<br/>Neo4j Relationships]
    GRAPH_RAG --> MEMORY[Memory Node<br/>Summarization]
    MEMORY --> ENTITY[Entity Node<br/>Extraction]
    ENTITY --> GENERATE[Generate Node<br/>LLM Call]
    
    GENERATE --> REFLECT{Reflector}
    REFLECT -->|Low Quality| GENERATE
    REFLECT -->|Good| HITL[HITL Node<br/>Human Approval]
    
    HITL --> COST[Cost Node<br/>Tracking]
    COST --> OUTPUT
    OUTPUT --> END([End])
```

### Key Concepts

1. **Nodes** - Each box is a function that processes state
2. **Edges** - Arrows show the flow between nodes
3. **Conditional Edges** - Diamonds represent decisions (like cache hit/miss)
4. **State** - A shared dictionary passed through all nodes

In [ ]:
# Let's look at the agent's compiled graph
print("Graph Nodes:")
for node_name in agent.graph.nodes:
    print(f"  📦 {node_name}")

### The State Object

Every node reads from and writes to a shared **State** dictionary:

In [ ]:
# View the state definition
import typing

from app.agents.demo.state import DemoAgentState

print("State Fields:")
for field_name, field_type in typing.get_type_hints(DemoAgentState).items():
    print(f"  • {field_name}: {field_type}")

---

# Chapter 3: Safety Guardrails 🛡️

**Learning Objectives:**
- Enable PII detection and masking
- Understand content moderation
- See how unsafe content is handled

Safety is a first-class citizen in LegendStack.

In [ ]:
# Create an agent with safety features emphasized
safe_agent = LegendDemoAgent(
    config=DemoAgentConfig(
        ENABLE_PII_GUARD=True,
        ENABLE_MODERATION=True,
    )
)

print("🛡️ Safety agent configured!")

In [ ]:
# Test PII Detection
pii_query = "My email is john.doe@company.com and my phone is 555-123-4567"

result = await safe_agent.chat(pii_query, thread_id="safety-demo")

print("Original Input:", pii_query)
print()
print("Features Used:", result["features_used"])
print()
if "pii_masking" in result["features_used"]:
    print("✅ PII was detected and masked!")

### PII Types Detected

| Pattern | Example | Masked As |
|---------|---------|----------|
| Email | user@example.com | [EMAIL_MASKED] |
| Phone | 555-123-4567 | [PHONE_MASKED] |
| IP Address | 192.168.1.1 | [IP_MASKED] |
| SSN | 123-45-6789 | [SSN_MASKED] |
| Credit Card | 4111-1111-1111-1111 | [CC_MASKED] |

In [ ]:
# Let's use the InputNode directly to see masking in action
from app.agents.demo.nodes import InputNode

input_node = InputNode(DemoAgentConfig())

# Test state
test_state = {
    "messages": [{"role": "user", "content": "Contact me at secret@company.com or 555-867-5309"}],
    "metadata": {},
}

result = await input_node(test_state)

print("Before:", test_state["messages"][-1]["content"])
print("After: ", result["sanitized_input"])

---

# Chapter 4: RAG Deep Dive 📚

**Learning Objectives:**
- Understand Retrieval-Augmented Generation
- See how vector search works
- Examine retrieved context

RAG grounds AI responses in your actual data.

In [ ]:
# Import the RAG components
from app.agents.demo.mocks import MockLLM, MockVectorStore
from app.agents.demo.nodes import RAGNode

# Create instances
vector_store = MockVectorStore()
llm = MockLLM()

print("📚 Vector Store Documents:")
for doc in vector_store.documents[:3]:
    print(f"  [{doc['id']}] {doc['content'][:60]}...")

In [ ]:
# Perform a similarity search
query = "How does semantic caching work?"
query_embedding = await llm.get_embeddings(query)

results = await vector_store.similarity_search(query_embedding, k=3)

print(f"Query: {query}\n")
print("Top 3 Results:")
for i, doc in enumerate(results, 1):
    print(f"  {i}. [Score: {doc['score']:.3f}] {doc['content'][:80]}...")

### How RAG Works

```
User Query → Embed → Vector Search → Context → LLM → Response
     ↓           ↓          ↓           ↓        ↓
 "How does..."  [0.2, 0.5...]  Top K docs  "Based on..."  "RAG combines..."
```

In [ ]:
# See the full RAG node in action
rag_node = RAGNode(config=DemoAgentConfig(), vector_store=vector_store, llm=llm)

state = {
    "sanitized_input": "What is entity memory?",
    "cache_hit": False,
}

result = await rag_node(state)

print("Retrieved Context:")
print(result["context"])

---

# Chapter 5: Graph-RAG 🔗

**Learning Objectives:**
- Understand knowledge graph integration
- See entity relationships
- Combine vector + graph retrieval

Graph-RAG enhances context with **relationships** that vector search alone can miss.

In [ ]:
# Import the Graph-RAG components
from app.agents.demo.mocks import MockGraphDB

graph_db = MockGraphDB()

print("🔗 Knowledge Graph Entities:")
for name, node in list(graph_db.nodes.items())[:5]:
    print(f"  • {name}: {node}")

In [ ]:
# Query the graph for relationships
results = await graph_db.execute_query("MATCH (n {name: $name}) RETURN n", {"name": "legendstack"})

print("Entity: LegendStack")
print("Related Entities:")
for result in results:
    for rel in result.get("related", []):
        print(f"  → {rel['relationship']} → {rel['name']} ({rel['label']})")

### Graph-RAG Architecture

```
Query: "Tell me about LegendStack"
       ↓
┌──────────────────────────────────────┐
│  Vector Search (RAG)                 │
│  "LegendStack is a framework..."     │
└──────────────────────────────────────┘
       ↓
┌──────────────────────────────────────┐
│  Graph Expansion (Graph-RAG)         │
│  LegendStack → USES → FastAPI        │
│  LegendStack → USES → LangGraph      │
│  LegendStack → STORES_IN → Neo4j     │
└──────────────────────────────────────┘
       ↓
  Enriched Context for LLM
```

---

# Chapter 6: Memory & Entities 🧠

**Learning Objectives:**
- Understand conversation memory
- See entity extraction in action
- Learn about cross-thread persistence

Memory lets agents remember context across conversations.

In [ ]:
# Import entity extraction
from app.agents.demo.nodes import DemoEntityNode

entity_node = DemoEntityNode(config=DemoAgentConfig(ENABLE_ENTITY_MEMORY=True), graph_db=MockGraphDB())

# Test entity extraction
state = {
    "sanitized_input": "My name is Alice and I work on Project Phoenix at Acme Corp",
    "context": "",
    "cache_hit": False,
    "tenant_id": "demo",
}

result = await entity_node(state)

print("🧠 Extracted Entities:")
for entity in result.get("entities", []):
    print(f"  • {entity['name']} ({entity['type']})")

### Entity Types

| Type | Examples |
|------|----------|
| Person | Alice, John Smith |
| Project | Project Phoenix, Alpha |
| Organization | Acme Corp, Google |

---

# Chapter 7: Self-Correction 🔄

**Learning Objectives:**
- Understand the Reflector pattern
- See quality scoring in action
- Learn about auto-regeneration

The Reflector evaluates responses and triggers regeneration if quality is too low.

In [ ]:
# Import the Reflector
from app.agents.demo.nodes import ReflectorNode

reflector = ReflectorNode(DemoAgentConfig(ENABLE_REFLECTOR=True))

# Test a good response
good_state = {
    "response": "Based on the context, LegendStack specifically provides RAG capabilities because it integrates vector search with LLM generation.",
    "context": "LegendStack is an AI framework with RAG, Graph-RAG, and memory.",
    "cache_hit": False,
}

result = await reflector(good_state)
print(f"Good Response Score: {result['reflection']['score']:.2f}")
print(f"Needs Reflection: {result['reflection']['needed']}")

In [ ]:
# Test a poor response
poor_state = {
    "response": "I don't know.",
    "context": "LegendStack is an AI framework with RAG, Graph-RAG, and memory.",
    "cache_hit": False,
}

result = await reflector(poor_state)
print(f"Poor Response Score: {result['reflection']['score']:.2f}")
print(f"Needs Reflection: {result['reflection']['needed']}")
if result["reflection"]["critique"]:
    print(f"Critique: {result['reflection']['critique']}")

---

# Chapter 8: Build Your Agent 🛠️

**Learning Objectives:**
- Create a custom node
- Modify the agent flow
- Toggle features via configuration

Now let's build something custom!

### Step 1: Create a Custom Node

Every node follows this pattern:

```python
class MyCustomNode:
    def __init__(self, config):
        self.config = config
    
    async def __call__(self, state):
        # Read from state
        input_data = state.get("some_key")
        
        # Process
        output_data = process(input_data)
        
        # Return updates to state
        return {"new_key": output_data}
```

In [ ]:
# Let's create a simple "Sentiment" node
class SentimentNode:
    """Adds basic sentiment detection to the agent."""

    POSITIVE_WORDS = ["great", "good", "excellent", "love", "happy", "thanks"]
    NEGATIVE_WORDS = ["bad", "hate", "terrible", "awful", "angry", "frustrated"]

    def __init__(self, config):
        self.config = config

    async def __call__(self, state):
        text = state.get("sanitized_input", "").lower()

        pos_count = sum(1 for w in self.POSITIVE_WORDS if w in text)
        neg_count = sum(1 for w in self.NEGATIVE_WORDS if w in text)

        if pos_count > neg_count:
            sentiment = "positive"
        elif neg_count > pos_count:
            sentiment = "negative"
        else:
            sentiment = "neutral"

        return {
            "metadata": {
                **state.get("metadata", {}),
                "sentiment": sentiment,
            }
        }


# Test it!
sentiment_node = SentimentNode(DemoAgentConfig())

test_state = {"sanitized_input": "I love this product, it's excellent!", "metadata": {}}
result = await sentiment_node(test_state)

print(f"Detected Sentiment: {result['metadata']['sentiment']}")

### Step 2: Customize Configuration

Toggle features on/off to create specialized agents:

In [ ]:
# Create a minimal agent (fast, no frills)
minimal_config = DemoAgentConfig(
    USE_MOCKS=True,
    ENABLE_RAG=False,
    ENABLE_GRAPH_RAG=False,
    ENABLE_ENTITY_MEMORY=False,
    ENABLE_REFLECTOR=False,
    ENABLE_HITL=False,
)

minimal_agent = LegendDemoAgent(config=minimal_config)

print("Minimal Agent Config:")
for k, v in minimal_agent.get_config_summary().items():
    if not v:
        print(f"  ❌ {k}")

---

# 🎉 Congratulations!

You've completed the LegendStack tutorial!

## What You Learned

1. ✅ Created and used the Demo Agent
2. ✅ Understood LangGraph workflow structure
3. ✅ Implemented safety guardrails
4. ✅ Built RAG and Graph-RAG pipelines
5. ✅ Added memory and entity tracking
6. ✅ Configured self-correction
7. ✅ Created your own custom node

## Next Steps

- 📖 Read the [full documentation](https://legendstack.github.io/agentic-fastapi-template/)
- 💬 Join the [Discord community](https://discord.gg/legendstack)
- ⭐ Star the [GitHub repo](https://github.com/LegendStack/agentic-fastapi-template)
- 🛠️ Build your own agent!